## GOLD 02. CARGA TABLAS

## 1. Configuración

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Gold_Carga") \
    .getOrCreate()

# Trabajar en UTC para que los cambios de hora no dupliquen ni salten horas
spark.conf.set("spark.sql.session.timeZone", "UTC")


ESQUEMA = "gold"
RUTA_FESTIVOS ="silver_festivos"
RUTA_ESTACIONES = "silver_estaciones"
RUTA_AIRE ="silver_aire_horario"
RUTA_METEO = "silver_meteo"
RUTA_TRAFICO = "silver_trafico"
RUTA_ESPIRA = "silver_espira_distrito"
RUTA_ESTACION_DISTRITO = "silver_estacion_distrito"
RUTA_POBLACION = "silver_poblacion"
RUTA_VERDE = "silver_verde"
RUTA_RUIDO = 'silver_ruido'
RUTA_SONOMETRO_DISTRITO = 'silver_sonometros_distritos'


# Leemos la silver desde el bucket y las registramos como vistas
rutas = {
    "silver_aire_horario": "gs://tfm_unir1/processed/aire/eea/horario",
    "silver_meteo": "gs://tfm_unir1/processed/aire/gva",
    "silver_estaciones": "gs://tfm_unir1/processed/estaciones",
    "silver_trafico": "gs://tfm_unir1/processed/trafico/espiras",
    "silver_espira_distrito": "gs://tfm_unir1/processed/espira_distrito",
    "silver_estacion_distrito": "gs://tfm_unir1/processed/estaciones_distrito",
    "silver_poblacion": "gs://tfm_unir1/processed/poblacion_distrito",
    "silver_verde": "gs://tfm_unir1/processed/verde_distrito",
    "silver_ruido": "gs://tfm_unir1/processed/ruido/sonometros",
    "silver_sonometros_distritos": "gs://tfm_unir1/processed/sonometros_distritos",

}

for nombre, ruta in rutas.items():
        spark.read.parquet(ruta).createOrReplaceTempView (nombre)
        
#Festivos y eventos (CSV)
(spark.read.option ("header", True).option("inferSchema", True)
    .csv ("gs://tfm_unir1/festivos/festivos_eventos.csv")
    .createOrReplaceTempView ("silver_festivos"))

spark.sql("SHOW VIEWS").show()


# Funciones auxiliares: devuelven el nombre completo de la tabla y su ubicación
def tabla(nombre):
    return f"{ESQUEMA}.{nombre}" if ESQUEMA else nombre


26/11/24 21:22:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/11/24 21:22:18 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+--------------------+-----------+
|namespace|            viewName|isTemporary|
+---------+--------------------+-----------+
|         | silver_aire_horario|       true|
|         |silver_espira_dis...|       true|
|         |silver_estacion_d...|       true|
|         |   silver_estaciones|       true|
|         |     silver_festivos|       true|
|         |        silver_meteo|       true|
|         |    silver_poblacion|       true|
|         |        silver_ruido|       true|
|         |silver_sonometros...|       true|
|         |      silver_trafico|       true|
|         |        silver_verde|       true|
+---------+--------------------+-----------+



## 2. Carga de dim_tiempo

In [5]:
#spark.read.option("header", True).csv(RUTA_FESTIVOS).createOrReplaceTempView("calendario")

from pyspark.sql import functions as F

# Creamos vista para generar las horas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW horas AS
    SELECT EXPLODE(SEQUENCE(
        TIMESTAMP '2013-01-01 00:00:00',
        TIMESTAMP '2027-12-31 23:00:00',
        INTERVAL 1 HOUR)) AS fecha_hora
""")

# Creamos vista para cargar los festivos
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW calendario AS
    SELECT TO_DATE(fecha) AS fecha, tipo, MIN(nombre) AS nombre
    FROM silver_festivos
    GROUP BY TO_DATE(fecha), tipo

""")

# Carga de dim_tiempo usando como join las views horas y calendario
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_tiempo")}
    SELECT
        CAST(DATE_FORMAT(h.fecha_hora, 'yyyyMMddHH') AS INT) AS id_tiempo,
        h.fecha_hora,
        TO_DATE(h.fecha_hora) AS fecha,
        YEAR(h.fecha_hora) AS anio,
        MONTH(h.fecha_hora) AS mes,
        DAY(h.fecha_hora) AS dia,
        HOUR(h.fecha_hora) AS hora,
        DAYOFWEEK(h.fecha_hora) IN (1,7) AS es_fin_de_semana,
        f.nombre IS NOT NULL AS es_festivo,
        f.nombre AS nombre_festivo,
        e.fecha IS NOT NULL AS es_evento,
        e.nombre AS nombre_evento,
        TO_DATE(h.fecha_hora) BETWEEN '2020-03-14' AND '2020-06-21' AS es_confinamiento,
        ((DAYOFWEEK(h.fecha_hora) + 5) % 7) + 1 AS dia_semana,
        ELEMENT_AT(ARRAY('lunes','martes','miércoles','jueves','viernes','sábado','domingo'),
                   ((DAYOFWEEK(h.fecha_hora) + 5) % 7) + 1) AS nombre_dia,
        ELEMENT_AT(ARRAY('enero','febrero','marzo','abril','mayo','junio','julio',
                         'agosto','septiembre','octubre','noviembre','diciembre'),
                   MONTH(h.fecha_hora)) AS nombre_mes,
        QUARTER(h.fecha_hora) AS trimestre
    FROM horas h
    LEFT JOIN calendario f ON TO_DATE(h.fecha_hora) = TO_DATE(f.fecha) AND f.tipo = 'festivo'
    LEFT JOIN calendario e ON TO_DATE(h.fecha_hora) = TO_DATE(e.fecha) AND e.tipo = 'evento'

""")

spark.sql(f"SELECT COUNT(*) AS horas, COUNT(DISTINCT id_tiempo) AS ids_unicos FROM {tabla('dim_tiempo')}").show()
    


26/11/21 14:44:28 WARN DAGScheduler: Broadcasting large task binary with size 1312.3 KiB


+------+----------+
| horas|ids_unicos|
+------+----------+
|131472|    131472|
+------+----------+



In [6]:
spark.sql(f"SELECT fecha, dia_semana, nombre_dia, nombre_mes, trimestre FROM {tabla('dim_tiempo')} WHERE hora = 0 LIMIT 7").show()

+----------+----------+----------+----------+---------+
|     fecha|dia_semana|nombre_dia|nombre_mes|trimestre|
+----------+----------+----------+----------+---------+
|2013-01-01|         2|    martes|     enero|        1|
|2013-01-02|         3| miércoles|     enero|        1|
|2013-01-03|         4|    jueves|     enero|        1|
|2013-01-04|         5|   viernes|     enero|        1|
|2013-01-05|         6|    sábado|     enero|        1|
|2013-01-06|         7|   domingo|     enero|        1|
|2013-01-07|         1|     lunes|     enero|        1|
+----------+----------+----------+----------+---------+



## 3. Carga de dim_estacion

In [7]:
# Creamos vista para cargar las estaciones desde la capa silver
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estaciones AS
    SELECT * FROM {RUTA_ESTACIONES}
""")


# Creamos vista con el distrito de cada estacion (calculado en QGIS)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estaciones_distrito AS
    SELECT * FROM {RUTA_ESTACION_DISTRITO}
""")


# Carga de dim_estacion. Agrego dos estaciones faltantes a mano
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_estacion")}
    SELECT e.id_estacion, e.codigo_eea, e.nombre_estacion, e.tipo_estacion, e.zona,
        e.latitud, e.longitud, e.altitud, e.fecha_alta, e.fecha_baja, e.activa,
        d.id_distrito   
    FROM estaciones e
    LEFT JOIN estaciones_distrito d ON e.id_estacion = d.id_estacion
    
    UNION ALL
    -- Estaciones solo meteorologicas (GVA)
    SELECT * FROM VALUES
        ('46250049', NULL, 'CONSELLERIA METEO.', 'Meteorológica', 'Urbana',
         NULL, NULL, NULL, NULL, NULL, TRUE, 7),
        ('46250900', NULL, 'NAZARET MET-2', 'Meteorológica', 'Urbana',
         NULL, NULL, NULL, NULL, NULL, TRUE, 11)
    
""")

spark.sql(f"SELECT * FROM {tabla('dim_estacion')} ORDER BY id_estacion").show(truncate=False)    

+-----------+----------+------------------+-------------+---------+--------+--------+-------+----------+----------+------+-----------+
|id_estacion|codigo_eea|nombre_estacion   |tipo_estacion|zona     |latitud |longitud|altitud|fecha_alta|fecha_baja|activa|id_distrito|
+-----------+----------+------------------+-------------+---------+--------+--------+-------+----------+----------+------+-----------+
|46250030   |ES1239A   |PISTA DE SILLA    |Tráfico      |Urbana   |39.45611|-0.37583|11.0   |1992-07-01|null      |true  |10         |
|46250043   |ES1619A   |VIVERS            |Fondo        |Urbana   |39.47806|-0.36833|11.0   |2002-01-01|null      |true  |5          |
|46250046   |ES1885A   |POLITÈCNIC        |Fondo        |Suburbana|39.4803 |-0.3364 |7.0    |2008-04-04|null      |true  |13         |
|46250047   |ES1912A   |AVD. FRANCIA      |Tráfico      |Urbana   |39.4575 |-0.34278|7.0    |2009-01-01|null      |true  |12         |
|46250048   |ES1926A   |MOLÍ DEL SOL      |Tráfico     

## 4. Carga de dim_contaminante

In [8]:
# dim_contaminante

# Carga de dim_contaminante

spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_contaminante")}
    SELECT * FROM VALUES
        (8,'NO2', 'µg/m³', 'Media anual', 40.0, 20.0, 10.0),
        (5,'PM10', 'µg/m³', 'Media anual', 40.0, 20.0, 15.0),
        (6001,'PM2.5', 'µg/m³', 'Media anual', 25.0, 10.0, 5.0), 
        (7,'O3', 'µg/m³', 'Dias al año con media de 8h > 120 µg/m³', 25.0, 18.0, NULL)
    
""")

spark.sql(f"SELECT * FROM {tabla('dim_contaminante')}").show(truncate=False)

+---------------+------+------+---------------------------------------+----------------+----------------+-------------+
|id_contaminante|nombre|unidad|indicador                              |limite_ue_actual|objetivo_ue_2030|guia_oms_2021|
+---------------+------+------+---------------------------------------+----------------+----------------+-------------+
|6001           |PM2.5 |µg/m³ |Media anual                            |25.0            |10.0            |5.0          |
|7              |O3    |µg/m³ |Dias al año con media de 8h > 120 µg/m³|25.0            |18.0            |null         |
|8              |NO2   |µg/m³ |Media anual                            |40.0            |20.0            |10.0         |
|5              |PM10  |µg/m³ |Media anual                            |40.0            |20.0            |15.0         |
+---------------+------+------+---------------------------------------+----------------+----------------+-------------+



- UE actual: Directiva 2008/50/CE.
- UE 2030: Directiva (UE) 2024/2881, que son los objetivos que usa el dashboard.
- OMS 2021: guías de calidad del aire de la OMS.

## 5. Carga de dim_distrito

In [9]:
# Vista con la población por distrito y año
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW poblacion AS
    SELECT * FROM {RUTA_POBLACION}
""")

# Carga de dim_distrito. La superficie se añade a mano (Anuario Estadístico 2025, Ajuntament de València)
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_distrito")}
    SELECT p.id_distrito, p.distrito, s.superficie_km2, e.latitud, e.longitud
    FROM (SELECT DISTINCT id_distrito, distrito FROM poblacion) p
    LEFT JOIN (SELECT * FROM VALUES
        (1, 1.690), (2, 1.733), (3, 1.972), (4, 5.238), (5, 1.944),
        (6, 1.693), (7, 1.989), (8, 2.873), (9, 2.985), (10, 11.325),
        (11, 3.967), (12, 2.367), (13, 2.959), (14, 1.570), (15, 2.629),
        (16, 2.216), (17, 15.196), (18, 2.011), (19, 32.426)
        AS t(id_distrito, superficie_km2)) s
    ON p.id_distrito = s.id_distrito
    LEFT JOIN (SELECT id_distrito, AVG(latitud) AS latitud, AVG (longitud) AS longitud
                FROM silver_espira_distrito
                GROUP BY id_distrito) e
    ON p.id_distrito = e.id_distrito
""")

spark.sql(f"SELECT * FROM {tabla('dim_distrito')} ORDER BY id_distrito").show(truncate=False)

+-----------+-----------------+--------------+------------------+--------------------+
|id_distrito|distrito         |superficie_km2|latitud           |longitud            |
+-----------+-----------------+--------------+------------------+--------------------+
|1          |CIUTAT VELLA     |1.69          |39.47293401260492 |-0.3764560744568048 |
|2          |L'EIXAMPLE       |1.733         |39.46421511200184 |-0.3697071773063341 |
|3          |EXTRAMURS        |1.972         |39.46962876042796 |-0.38698243505418295|
|4          |CAMPANAR         |5.238         |39.482072964757116|-0.3970298738197626 |
|5          |LA SAIDIA        |1.944         |39.48559876526005 |-0.3748054479024884 |
|6          |EL PLA DEL REAL  |1.693         |39.47448535481013 |-0.36020420716575624|
|7          |L'OLIVERETA      |1.989         |39.46861741492662 |-0.40327992006242536|
|8          |PATRAIX          |2.873         |39.46036575434315 |-0.39833414725153937|
|9          |JESUS            |2.985       

## 6. Carga de dim_espira

In [10]:
# Vista con las espiras y su distrito (calculado en QGIS)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW espiras AS
    SELECT * FROM {RUTA_ESPIRA}
""")

# Carga de dim_espira
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_espira")}
    SELECT id_espira, latitud, longitud, id_distrito
    FROM espiras
""")

spark.sql(f"""
    SELECT COUNT(*) AS espiras,
           COUNT(DISTINCT id_espira) AS espiras_distintas,
           SUM(CASE WHEN id_distrito IS NULL THEN 1 ELSE 0 END) AS sin_distrito
    FROM {tabla("dim_espira")}
""").show()

+-------+-----------------+------------+
|espiras|espiras_distintas|sin_distrito|
+-------+-----------------+------------+
|    964|              964|           0|
+-------+-----------------+------------+



## 7. Carga de dim_sonometro

In [4]:
# Carga de dim_sonometro: coordenadas y distrito (QGIS) + red (silver de ruido)
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("dim_sonometro")}
    SELECT d.id_sensor                  AS id_sonometro,
           r.red,
           d.latitud,
           d.longitud,
           CAST(d.coddistrit AS INT)    AS id_distrito
    FROM {RUTA_SONOMETRO_DISTRITO} d
    JOIN (SELECT DISTINCT id_sensor, red FROM {RUTA_RUIDO}) r
      ON d.id_sensor = r.id_sensor
""")

spark.sql(f"""
    SELECT COUNT(*) AS sonometros,
           COUNT(DISTINCT id_sonometro) AS sonometros_distintos,
           SUM(CASE WHEN id_distrito IS NULL THEN 1 ELSE 0 END) AS sin_distrito
    FROM {tabla("dim_sonometro")}
""").show()

+----------+--------------------+------------+
|sonometros|sonometros_distintos|sin_distrito|
+----------+--------------------+------------+
|       137|                 137|           0|
+----------+--------------------+------------+



## 8. Carga de fact_aire

In [11]:
# fact_aire

# Creamos vista para cargar los datos desde la capa silver
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW aire_horario AS
    SELECT * FROM {RUTA_AIRE}
""")

spark.sql(f"""
    SELECT id_punto_medicion,
        REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1) AS id_estacion,
        COUNT(*) AS registros
        
    FROM aire_horario
    GROUP BY id_punto_medicion
    ORDER BY id_punto_medicion
""").show(50,truncate=False)

# Carga de fact_aire
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_aire")}
    SELECT  a.id_tiempo, a.id_estacion, a.id_contaminante,
            e.id_distrito,
            a.concentracion
    FROM (    
        SELECT
            -- La EEA da las horas en UTC+1 fijo: se resta 1 h (a UTC) y se pasa a hora de Valencia
            CAST(DATE_FORMAT(FROM_UTC_TIMESTAMP(fecha_inicio - INTERVAL 1 HOUR, 'Europe/Madrid'), 'yyyyMMddHH') AS INT) AS id_tiempo,
            REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1)  AS id_estacion,
            codigo_contaminante                                   AS id_contaminante,
            AVG(CAST(concentracion AS DOUBLE))                    AS concentracion
        FROM aire_horario
        WHERE concentracion IS NOT NULL
            AND REGEXP_EXTRACT(id_punto_medicion, 'SP_([0-9]+)_', 1) LIKE '46250%'   -- solo estaciones de Valencia ciudad (excluimos 46102002, 46078004, 46190005, 46244003)
        GROUP BY 1, 2, 3
    ) a
    LEFT JOIN {tabla("dim_estacion")} e ON a.id_estacion = e.id_estacion
""")

spark.sql(f"SELECT * FROM {tabla('fact_aire')}").show(truncate=False)

+--------------------+-----------+---------+
|id_punto_medicion   |id_estacion|registros|
+--------------------+-----------+---------+
|ES/SP_46078004_14_6 |46078004   |112876   |
|ES/SP_46078004_8_8  |46078004   |107468   |
|ES/SP_46102002_10_46|46102002   |110253   |
|ES/SP_46102002_14_6 |46102002   |112707   |
|ES/SP_46102002_8_8  |46102002   |108971   |
|ES/SP_46102002_9_46 |46102002   |110249   |
|ES/SP_46190005_14_6 |46190005   |109567   |
|ES/SP_46190005_8_8  |46190005   |106058   |
|ES/SP_46244003_10_46|46244003   |27024    |
|ES/SP_46244003_14_6 |46244003   |70056    |
|ES/SP_46244003_8_8  |46244003   |61296    |
|ES/SP_46244003_9_46 |46244003   |27024    |
|ES/SP_46250030_10_46|46250030   |104325   |
|ES/SP_46250030_14_6 |46250030   |110663   |
|ES/SP_46250030_8_8  |46250030   |110593   |
|ES/SP_46250030_9_46 |46250030   |104319   |
|ES/SP_46250043_14_6 |46250043   |111335   |
|ES/SP_46250043_8_8  |46250043   |107104   |
|ES/SP_46250046_10_46|46250046   |113090   |
|ES/SP_462

+----------+-----------+---------------+-----------+-------------+
|id_tiempo |id_estacion|id_contaminante|id_distrito|concentracion|
+----------+-----------+---------------+-----------+-------------+
|2013011820|46250048   |7              |4          |44.0         |
|2013020813|46250048   |7              |4          |60.0         |
|2013021506|46250048   |7              |4          |1.0          |
|2013022317|46250048   |7              |4          |68.0         |
|2013022611|46250048   |7              |4          |52.0         |
|2013030205|46250048   |7              |4          |2.0          |
|2013030310|46250048   |7              |4          |37.0         |
|2013032118|46250048   |7              |4          |45.0         |
|2013032221|46250048   |7              |4          |64.0         |
|2013040318|46250048   |7              |4          |58.0         |
|2013040621|46250048   |7              |4          |71.0         |
|2013041720|46250048   |7              |4          |56.0      

In [13]:
# Comprobamos que todas las mediciones tengan estacion
spark.sql(f"""
    SELECT COUNT(*)                                                AS registros,
           SUM(CASE WHEN e.id_estacion IS NULL THEN 1 ELSE 0 END)  AS sin_estacion,
           SUM(CASE WHEN t.id_tiempo   IS NULL THEN 1 ELSE 0 END)  AS sin_tiempo,
           SUM(CASE WHEN a.id_distrito   IS NULL THEN 1 ELSE 0 END)  AS sin_distrito
    FROM {tabla("fact_aire")} a
    LEFT JOIN {tabla("dim_estacion")} e ON a.id_estacion = e.id_estacion
    LEFT JOIN {tabla("dim_tiempo")}   t ON a.id_tiempo   = t.id_tiempo
""").show()

+---------+------------+----------+------------+
|registros|sin_estacion|sin_tiempo|sin_distrito|
+---------+------------+----------+------------+
|  2215570|           0|         0|           0|
+---------+------------+----------+------------+



In [14]:
# Chequeamos que las horas hayan quedado bien cargadas
spark.sql(f"""
    SELECT d.hora, ROUND(AVG(a.concentracion), 1) AS no2_medio
    FROM {tabla('fact_aire')} a
    JOIN {tabla('dim_tiempo')} d ON a.id_tiempo = d.id_tiempo
    JOIN {tabla('dim_contaminante')} c ON a.id_contaminante = c.id_contaminante
    WHERE (c.nombre LIKE '%NO2%' OR a.id_contaminante = 8)
      AND NOT d.es_fin_de_semana
    GROUP BY d.hora ORDER BY d.hora
""").show(24)

+----+---------+
|hora|no2_medio|
+----+---------+
|   0|     20.1|
|   1|     17.8|
|   2|     15.7|
|   3|     14.4|
|   4|     14.1|
|   5|     16.1|
|   6|     23.8|
|   7|     36.4|
|   8|     43.4|
|   9|     38.3|
|  10|     29.4|
|  11|     24.3|
|  12|     21.7|
|  13|     20.4|
|  14|     19.1|
|  15|     18.3|
|  16|     19.4|
|  17|     23.5|
|  18|     27.7|
|  19|     30.9|
|  20|     31.9|
|  21|     30.9|
|  22|     28.5|
|  23|     24.6|
+----+---------+



La EEA publica las horas en UTC+1 fijo; se convirtieron a hora local de València (Europe/Madrid). Se validó comprobando que el pico de NO₂ coincide con la hora punta del tráfico (8 h).

## 8. Carga de fact_meteo

In [15]:
# Creamos vista para cargar los datos desde la capa silver
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW meteo AS
    SELECT * FROM {RUTA_METEO}
""")


# Carga de fact_meteo
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_meteo")}
    SELECT
       CAST(DATE_FORMAT(datetime, 'yyyyMMddHH') AS INT) AS id_tiempo,
       station_code AS id_estacion,
       temperature_clean AS temperatura,
       relative_humidity_clean AS humedad_relativa,
       wind_speed_clean AS velocidad_viento,
       wind_direction_deg_clean AS direccion_viento,
       precipitation_clean AS precipitacion,
       pressure_clean AS presion,
       solar_radiation_clean AS radiation_solar
    FROM meteo
    WHERE COALESCE(temperature_clean, relative_humidity_clean, wind_speed_clean,
                   wind_direction_deg_clean, precipitation_clean, pressure_clean,
                   solar_radiation_clean) IS NOT NULL
""")

spark.sql(f"SELECT * FROM {tabla('fact_meteo')}").show(truncate=False)

+----------+-----------+-----------+----------------+----------------+----------------+-------------+-------+---------------+
|id_tiempo |id_estacion|temperatura|humedad_relativa|velocidad_viento|direccion_viento|precipitacion|presion|radiacion_solar|
+----------+-----------+-----------+----------------+----------------+----------------+-------------+-------+---------------+
|2022070100|46250047   |null       |null            |1.3             |68.0            |null         |null   |null           |
|2022070101|46250047   |null       |null            |0.5             |41.0            |null         |null   |null           |
|2022070102|46250047   |null       |null            |0.6             |29.0            |null         |null   |null           |
|2022070103|46250047   |null       |null            |0.4             |35.0            |null         |null   |null           |
|2022070104|46250047   |null       |null            |0.3             |22.0            |null         |null   |null     

In [16]:
spark.sql(f"""
    SELECT m.id_estacion, e.nombre_estacion, COUNT(*) AS registros,
           MIN(t.fecha) AS desde, MAX(t.fecha) AS hasta
    FROM {tabla("fact_meteo")} m
    LEFT JOIN {tabla("dim_estacion")} e ON m.id_estacion = e.id_estacion
    LEFT JOIN {tabla("dim_tiempo")}   t ON m.id_tiempo   = t.id_tiempo
    GROUP BY m.id_estacion, e.nombre_estacion
    ORDER BY m.id_estacion
""").show(truncate=False)

+-----------+------------------+---------+----------+----------+
|id_estacion|nombre_estacion   |registros|desde     |hasta     |
+-----------+------------------+---------+----------+----------+
|46250047   |AVD. FRANCIA      |51462    |2020-01-01|2025-12-31|
|46250049   |CONSELLERIA METEO.|51259    |2020-01-01|2025-12-31|
|46250900   |NAZARET MET-2     |47886    |2020-05-01|2025-12-31|
+-----------+------------------+---------+----------+----------+



## 9. Carga de fact_trafico

In [17]:
# fact_trafico
# Creamos vista para cargar los datos desde la capa silver
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW trafico AS
    SELECT * FROM {RUTA_TRAFICO}
""")


# Carga de fact_meteo
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_trafico")}
    SELECT
       CAST(DATE_FORMAT(t.fecha_inicio, 'yyyyMMddHH') AS INT) AS id_tiempo,
       CAST(t.id_espira AS STRING) AS id_espira,
       e.id_distrito,
       t.intensidad
    FROM trafico t
    JOIN {tabla("dim_espira")} e ON CAST(t.id_espira as STRING) = e.id_espira
    
""")

spark.sql(f"SELECT * FROM {tabla('fact_trafico')}").show(truncate=False)

+----------+---------+-----------+----------+
|id_tiempo |id_espira|id_distrito|intensidad|
+----------+---------+-----------+----------+
|2024012722|3210     |11         |0.0       |
|2024010410|3210     |11         |293.0     |
|2024010416|3606     |4          |1620.0    |
|2024011310|1436     |6          |627.0     |
|2024010422|1436     |6          |660.0     |
|2024011409|2904     |11         |0.0       |
|2024011102|3606     |4          |126.0     |
|2024011123|3210     |11         |0.0       |
|2024012613|3210     |11         |0.0       |
|2024011109|3606     |4          |3270.0    |
|2024011300|2904     |11         |0.0       |
|2024011310|3606     |4          |1925.0    |
|2024012623|3606     |4          |810.0     |
|2024010108|3606     |4          |209.0     |
|2024012407|3606     |4          |810.0     |
|2024010401|1436     |6          |168.0     |
|2024011411|3606     |4          |1548.0    |
|2024012703|2904     |11         |42.0      |
|2024011106|3210     |11         |

In [18]:
spark.sql(f"""
    SELECT COUNT(*)                                               AS registros,
           COUNT(DISTINCT f.id_espira)                              AS espiras,
           SUM(CASE WHEN t.id_tiempo IS NULL THEN 1 ELSE 0 END)   AS sin_tiempo,
           SUM(CASE WHEN f.id_distrito IS NULL THEN 1 ELSE 0 END)   AS sin_distrito
    FROM {tabla("fact_trafico")} f
    LEFT JOIN {tabla("dim_tiempo")} t ON f.id_tiempo = t.id_tiempo
""").show()

+---------+-------+----------+------------+
|registros|espiras|sin_tiempo|sin_distrito|
+---------+-------+----------+------------+
| 11637477|    964|         0|           0|
+---------+-------+----------+------------+



## 10. Carga de fact_poblacion

In [19]:
# fact_poblacion
# La vista "poblacion" ya se creó en la carga de dim_distrito
spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_poblacion")}
    SELECT id_distrito, anio, poblacion
    FROM poblacion
""")

spark.sql(f"""
    SELECT anio, COUNT(*) AS distritos, SUM(poblacion) AS total
    FROM {tabla("fact_poblacion")}
    GROUP BY anio
    ORDER BY anio
""").show()

+----+---------+------+
|anio|distritos| total|
+----+---------+------+
|2023|       19|809501|
|2024|       19|830606|
|2025|       19|844424|
+----+---------+------+



## 11. Carga de fact_zona_verde

In [20]:
# fact_zona_verde

# Creamos vista para cargar los m² verdes por distrito (calculados en QGIS)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW verde AS
    SELECT * FROM {RUTA_VERDE}
""")

spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_zona_verde")}
    SELECT id_distrito, m2_verde, n_espacios
    FROM verde
""")

spark.sql(f"SELECT * FROM {tabla('fact_zona_verde')} ORDER BY id_distrito").show()

+-----------+--------+----------+
|id_distrito|m2_verde|n_espacios|
+-----------+--------+----------+
|          1|198118.0|        50|
|          2|226437.0|        22|
|          3|154407.0|        36|
|          4|606097.0|        72|
|          5|444293.0|        45|
|          6|321283.0|        39|
|          7|270855.0|        48|
|          8|227682.0|        44|
|          9|175048.0|        44|
|         10|887558.0|        89|
|         11|435561.0|        77|
|         12|357326.0|        51|
|         13|237110.0|        56|
|         14| 81182.0|        34|
|         15|174740.0|        50|
|         16|257396.0|        55|
|         17| 45406.0|        23|
|         18|135951.0|        19|
|         19|181351.0|        42|
+-----------+--------+----------+



## 12. Carga de fact_ruido


In [6]:
# fact_ruido

# Carga de fact_ruido: un valor de ruido por sonometro y hora

spark.sql(f"""
    INSERT OVERWRITE TABLE {tabla("fact_ruido")}
    SELECT  t.id_tiempo,
            r.id_sensor     AS id_sonometro,
            s.id_distrito, 
            r.laeq         AS nivel_ruido_db
    FROM {RUTA_RUIDO} r
    JOIN {tabla("dim_tiempo")} t
        ON t.fecha_hora = r.fecha_inicio
    JOIN {tabla("dim_sonometro")} s
        ON s.id_sonometro = r.id_sensor
""")

spark.sql(f"""
    SELECT COUNT(*) AS registros,
           COUNT(DISTINCT id_sonometro) AS sonometros,
           SUM(CASE WHEN id_distrito IS NULL THEN 1 ELSE 0 END) AS sin_distrito
    FROM {tabla("fact_ruido")}
""").show()

+---------+----------+------------+
|registros|sonometros|sin_distrito|
+---------+----------+------------+
|   905090|       137|           0|
+---------+----------+------------+



## Control de cantidad de datos cargados

In [22]:
spark.sql(f"""
    SELECT 'fact_trafico'    AS tabla, COUNT(*) AS registros FROM {tabla("fact_trafico")}
    UNION ALL SELECT 'fact_meteo',       COUNT(*) FROM {tabla("fact_meteo")}
    UNION ALL SELECT 'fact_aire',        COUNT(*) FROM {tabla("fact_aire")}
    UNION ALL SELECT 'fact_poblacion',   COUNT(*) FROM {tabla("fact_poblacion")}
    UNION ALL SELECT 'fact_zona_verde',  COUNT(*) FROM {tabla("fact_zona_verde")}
    UNION ALL SELECT 'dim_tiempo',       COUNT(*) FROM {tabla("dim_tiempo")}
    UNION ALL SELECT 'dim_contaminante', COUNT(*) FROM {tabla("dim_contaminante")}
    UNION ALL SELECT 'dim_estacion',     COUNT(*) FROM {tabla("dim_estacion")}
    UNION ALL SELECT 'dim_espira',       COUNT(*) FROM {tabla("dim_espira")}
    UNION ALL SELECT 'dim_distrito',     COUNT(*) FROM {tabla("dim_distrito")}
""").show()

+----------------+---------+
|           tabla|registros|
+----------------+---------+
|    fact_trafico| 11637477|
|      fact_meteo|   150607|
|       fact_aire|  2215570|
|  fact_poblacion|       57|
| fact_zona_verde|       19|
|      dim_tiempo|   131472|
|dim_contaminante|        4|
|    dim_estacion|       11|
|      dim_espira|      964|
|    dim_distrito|       19|
+----------------+---------+

